# Exploratory Data Analysis - Predict Customer Churn

## 1. Setup and data loading

In [4]:
import csv
import pandas as pd

df = pd.read_csv('train.csv')
print(f"Loaded {len(df)} rows and {len(df.columns)} columns from train.csv")

df

Loaded 594194 rows and 21 columns from train.csv


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No
594192,594192,Female,0,No,No,32,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No


## 2. Data Overview 

## 3. Data Quality Assessment

## 4. Univariate Analysis

### 4.1 Target Variable — Churn Distribution

Before looking at individual features, we examine the target variable `Churn` to check for **class imbalance**. This affects which model and training strategy we choose.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid')
CHURN_COLORS = {'No': '#2196F3', 'Yes': '#F44336'}

churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100
ratio        = churn_counts['No'] / churn_counts['Yes']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#f9f9f9')

# ── Bar chart ──
colors = [CHURN_COLORS[k] for k in churn_counts.index]
bars   = axes[0].bar(churn_counts.index, churn_counts.values,
                     color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, (label, count) in zip(bars, churn_counts.items()):
    pct = churn_pct[label]
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + churn_counts.max() * 0.02,
                 f'{count:,}\n({pct:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Churn Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Churn', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
axes[0].set_ylim(0, churn_counts.max() * 1.28)
axes[0].annotate(f'Imbalance ratio  {ratio:.1f} : 1  (No Churn : Churn)',
                 xy=(0.5, 0.91), xycoords='axes fraction', ha='center',
                 fontsize=10, color='#555',
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff9c4', edgecolor='#bbb'))

# ── Pie chart ──
wedge_colors = [CHURN_COLORS[k] for k in churn_pct.index]
wedges, texts, autotexts = axes[1].pie(
    churn_pct.values,
    labels=[f'{k} Churn' for k in churn_pct.index],
    autopct='%1.1f%%',
    colors=wedge_colors,
    startangle=90,
    explode=[0, 0.07],
    textprops={'fontsize': 11}
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_fontsize(13)
axes[1].set_title('Churn Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: Churn Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observation:** The dataset is significantly imbalanced — roughly **3 out of 4 customers did not churn**. The minority class (Churn = Yes) makes up only ~27% of records. A naive model that always predicts "No Churn" would appear ~73% accurate, making **accuracy a useless metric** here. We will use **ROC-AUC and F1-score** instead, and apply class weighting during training.

### 4.2 Numerical Features

The three continuous features are `tenure` (months as customer), `MonthlyCharges`, and `TotalCharges`.

We use **histograms** to see the shape of each distribution and **boxplots** to spot outliers.

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[numerical_cols] = df[numerical_cols].apply(pd.to_numeric, errors='coerce')

HIST_COLORS  = ['#5C6BC0', '#EF7D48', '#4CAF50']   # indigo, orange, green
MEAN_COLOR   = '#E53935'   # red   — dashed line
MEDIAN_COLOR = '#00897B'   # teal  — solid line

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor('#f9f9f9')

for i, col in enumerate(numerical_cols):
    data       = df[col].dropna()
    mean_val   = data.mean()
    median_val = data.median()
    skew_val   = data.skew()
    q1, q3     = data.quantile(0.25), data.quantile(0.75)
    iqr        = q3 - q1

    # ── Histogram ──
    axes[0, i].hist(data, bins=40, color=HIST_COLORS[i], edgecolor='white', alpha=0.85)
    axes[0, i].axvline(mean_val,   color=MEAN_COLOR,   linestyle='--', linewidth=2,
                       label=f'Mean: {mean_val:.1f}')
    axes[0, i].axvline(median_val, color=MEDIAN_COLOR, linestyle='-',  linewidth=2,
                       label=f'Median: {median_val:.1f}')
    axes[0, i].legend(fontsize=9, framealpha=0.9)
    axes[0, i].text(0.97, 0.95, f'Skewness: {skew_val:+.2f}',
                    transform=axes[0, i].transAxes, ha='right', va='top',
                    fontsize=9, bbox=dict(boxstyle='round', facecolor='#fff9c4', alpha=0.95))
    axes[0, i].set_title(f'{col}  —  Distribution', fontsize=12, fontweight='bold')
    axes[0, i].set_xlabel(col, fontsize=10)
    axes[0, i].set_ylabel('Frequency', fontsize=10)

    # ── Boxplot ──
    axes[1, i].boxplot(
        data, vert=True, patch_artist=True,
        boxprops=dict(facecolor=HIST_COLORS[i], alpha=0.55),
        medianprops=dict(color=MEDIAN_COLOR, linewidth=2.5),
        meanprops=dict(marker='D', markerfacecolor=MEAN_COLOR, markersize=7),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=1.5),
        flierprops=dict(marker='o', alpha=0.2, markersize=3,
                        markerfacecolor=HIST_COLORS[i]),
        showmeans=True
    )
    axes[1, i].text(0.97, 0.97,
                    f'Min: {data.min():.0f}\nQ1:  {q1:.0f}\nMed: {median_val:.0f}'
                    f'\nQ3:  {q3:.0f}\nMax: {data.max():.0f}\nIQR: {iqr:.0f}',
                    transform=axes[1, i].transAxes, ha='right', va='top',
                    fontsize=9, bbox=dict(boxstyle='round', facecolor='#fff9c4', alpha=0.95))
    axes[1, i].set_title(f'{col}  —  Boxplot', fontsize=12, fontweight='bold')
    axes[1, i].set_ylabel(col, fontsize=10)

plt.suptitle('Numerical Features: Distribution & Spread', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("── Statistical Summary ──")
print(df[numerical_cols].describe().round(2))
print("\n── Skewness (>1 = right-skewed, <-1 = left-skewed) ──")
print(df[numerical_cols].skew().round(3))

**Observations:**
- **`tenure`** — Bimodal: two peaks visible (many very new customers, and many long-term ones). The red dashed mean and teal median are close together, confirming the distribution is roughly symmetric across both groups. New customers are likely the most volatile.
- **`MonthlyCharges`** — Wide, roughly uniform spread with a slight right skew. The gap between mean and median is small. This feature varies meaningfully across the dataset and should carry useful signal for the model.
- **`TotalCharges`** — Clear right skew (skewness > 1). Mean is pulled well above the median by a tail of high-value customers. This is expected: `TotalCharges ≈ tenure × MonthlyCharges`, so it is not independent of the other two features. The boxplot shows no extreme outliers — the spread is natural. **We will test dropping this feature in the ablation study.**

### 4.3 Categorical Features

We use **count plots (bar charts)** for all categorical features. This shows how values are spread — whether one category dominates, or if there is good variety.

In [ ]:
categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

# Color-code by feature type so patterns are immediately visible
DEMO_COL     = '#7986CB'   # Indigo  — who the customer is
SERVICE_COL  = '#4DB6AC'   # Teal    — what services they use
CONTRACT_COL = '#FF8A65'   # Orange  — how they pay / are contracted

COLOR_MAP = {
    'gender': DEMO_COL, 'SeniorCitizen': DEMO_COL,
    'Partner': DEMO_COL, 'Dependents': DEMO_COL,
    'PhoneService': SERVICE_COL, 'MultipleLines': SERVICE_COL,
    'InternetService': SERVICE_COL, 'OnlineSecurity': SERVICE_COL,
    'DeviceProtection': SERVICE_COL, 'TechSupport': SERVICE_COL,
    'StreamingTV': SERVICE_COL, 'StreamingMovies': SERVICE_COL,
    'Contract': CONTRACT_COL, 'PaperlessBilling': CONTRACT_COL,
    'PaymentMethod': CONTRACT_COL,
}

n_cols = 3
n_rows = -(-len(categorical_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
fig.patch.set_facecolor('#f9f9f9')
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    counts = df[col].value_counts()
    total  = counts.sum()
    color  = COLOR_MAP[col]

    bars = axes[i].bar(counts.index.astype(str), counts.values,
                       color=color, edgecolor='white', linewidth=1.2)
    axes[i].set_title(col, fontsize=12, fontweight='bold', color='#222')
    axes[i].set_ylabel('Count', fontsize=9)
    axes[i].tick_params(axis='x', rotation=25, labelsize=9)
    axes[i].set_ylim(0, counts.max() * 1.30)

    for bar, v in zip(bars, counts.values):
        pct = v / total * 100
        axes[i].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + total * 0.005,
                     f'{v:,}\n({pct:.0f}%)',
                     ha='center', va='bottom', fontsize=8, fontweight='bold')

for k in range(len(categorical_cols), len(axes)):
    axes[k].set_visible(False)

legend_elements = [
    Patch(facecolor=DEMO_COL,     label='Demographics  (who they are)'),
    Patch(facecolor=SERVICE_COL,  label='Services  (what they use)'),
    Patch(facecolor=CONTRACT_COL, label='Contract & Billing  (how they pay)'),
]
fig.legend(handles=legend_elements, loc='lower right',
           fontsize=11, framealpha=0.95, title='Feature Category', title_fontsize=11)

plt.suptitle('Categorical Features: Value Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- **Demographics (indigo):** `gender` is near 50/50 — unlikely to be a strong standalone predictor. `SeniorCitizen` is heavily skewed (~84% non-senior), meaning the model will have less data to learn from for senior customers.
- **Services (teal):** Most customers opted *out* of add-on services (`OnlineSecurity`, `TechSupport`, `DeviceProtection`, etc.). A large "No" proportion across these features may indicate less engaged or value-aware customers — worth investigating against churn in bivariate analysis.
- **Contract & Billing (orange):** `Contract` shows month-to-month as the dominant type (~55%). These customers have **no lock-in**, making them the most at risk. `PaymentMethod` shows electronic check as most common — a manual payment method often associated with higher churn. `PaperlessBilling` leans toward "Yes".
- **All features retained:** No feature is so skewed that it needs to be dropped at this stage. All 15 will be encoded and passed to the model; feature importance from training will confirm which are truly useful.

### 4.4 Summary of Univariate Findings & Modelling Implications

---

#### Target Variable (`Churn`)
The dataset is **class-imbalanced** — roughly **73% No Churn vs 27% Churn** (approximately a 3:1 ratio).

**Why this matters:** A model that blindly predicts "No Churn" for every customer would score ~73% accuracy — this is misleading and useless for a business. We must:
- Use **F1-score, ROC-AUC, or Precision-Recall AUC** as our evaluation metrics instead of accuracy
- Apply **`class_weight='balanced'`** during model training, or use **SMOTE** to oversample the minority class

---

#### Numerical Features (`tenure`, `MonthlyCharges`, `TotalCharges`)

| Feature | Shape | Mean vs Median | What it suggests |
|---|---|---|---|
| `tenure` | Bimodal (two peaks) | Close | Many new customers AND long-term ones — churn likely concentrated at low tenure |
| `MonthlyCharges` | Roughly uniform / slight right skew | Mean > Median | Wide spread; higher charges may signal churn risk |
| `TotalCharges` | Right-skewed | Mean >> Median | Mostly low totals; a few very large — largely a product of tenure × MonthlyCharges |

**Key insight:** When Mean > Median, the distribution is pulled right by a tail of high values. For `TotalCharges` this is strongest, meaning the majority of customers have accumulated low total charges (i.e., they are relatively new or cheap-plan customers).

**Action:** `TotalCharges` is mathematically derived from `tenure` × `MonthlyCharges`. We should test whether dropping it improves the model (ablation study). No outlier removal is needed — the spread is natural, not erroneous.

---

#### Categorical Features

| Feature | Observation | Implication |
|---|---|---|
| `gender` | Near 50/50 split | Balanced — unlikely to be a strong predictor on its own |
| `SeniorCitizen` | ~84% non-senior | Highly skewed — model may underfit senior customers |
| `Partner` / `Dependents` | Slight majority without | Customers without dependents may have less reason to stay |
| `Contract` | Month-to-month dominates (~55%) | No lock-in = highest churn risk; strongest single categorical signal |
| `InternetService` | Fiber optic + DSL dominant, "No" is minority | Fiber optic customers may pay more and churn more |
| `OnlineSecurity`, `TechSupport`, etc. | Majority opted out ("No") | Low engagement with add-ons may correlate with disengagement overall |
| `PaymentMethod` | Electronic check is most common | Manual payment methods may correlate with higher churn |
| `PaperlessBilling` | More Yes than No | Slight lean toward digital — pair with bivariate analysis |

---

#### Modelling Decisions from Univariate Analysis

1. **Metric:** Use **ROC-AUC** and **F1-score** — not accuracy — due to class imbalance
2. **Class imbalance handling:** Apply `class_weight='balanced'` (or SMOTE) in all models
3. **Feature candidates to watch:** `Contract`, `tenure`, `MonthlyCharges` appear most informative from shape alone
4. **Feature to test dropping:** `TotalCharges` may be redundant — investigate in pipeline ablation
5. **Model family:** Tree-based models (Random Forest, XGBoost, LightGBM) are the right starting point — they handle skewed numerics, categorical features, and class imbalance naturally without requiring feature scaling

## 5. Bivariate Analysis